# Public Trade Analysis

Loads raw trades from `data/trades_raw/`, grouped by **wallet × market × timestamp**,
and enriches each trade with the **final token value** (winner / loser / price at resolution).

In [1]:
import json
import datetime
from pathlib import Path

import pandas as pd

## Parameters

In [2]:
# ── date range (inclusive) of market folders to load ──────────────────────────
START_DATE     = datetime.date(2026, 3, 1)
END_DATE       = datetime.date(2026, 3, 1)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-5% wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 2, 1)

# ── path to trades_raw ────────────────────────────────────────────────────────
TRADES_RAW = Path("../data/trades_raw")

# ── output parquet ────────────────────────────────────────────────────────────
TRADES_PARQUET = Path("../data/trades_processed.parquet")

## 1 · Load markets and trades

In [3]:
import json
import datetime
from pathlib import Path

import pandas as pd

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

In [4]:
def load_market(json_path: Path) -> dict:
    """Load a single market definition from its .json file."""
    with json_path.open("rb") as f:
        return orjson.loads(f.read()) if "orjson" in dir() else json.load(f)


def load_trades(jsonl_path: Path) -> list[dict]:
    """Load all trades from a .jsonl file into a list of dicts."""
    rows = []
    with jsonl_path.open("rb") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json_loads(line)
            obj["asset"] = str(obj["asset"])
            rows.append(obj)
    return rows


def day_folders(root: Path, start: datetime.date, end: datetime.date) -> list[Path]:
    """Return sorted list of YYYY-MM-DD folders within [start, end]."""
    folders = []
    for p in sorted(root.iterdir()):
        if not p.is_dir():
            continue
        try:
            d = datetime.date.fromisoformat(p.name)
        except ValueError:
            continue
        if start <= d <= end:
            folders.append(p)
    return folders

In [5]:
folders = day_folders(TRADES_RAW, START_DATE, END_DATE)
print(f"Found {len(folders)} day-folder(s): {[f.name for f in folders]}")

Found 1 day-folder(s): ['2026-03-01']


In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

def load_market_and_trades(json_path: Path) -> tuple[dict, list[dict]] | None:
    """Load market definition and its trades from a single json/jsonl pair."""
    jsonl_path = json_path.with_suffix(".jsonl")
    if not jsonl_path.exists():
        return None
    
    market = load_market(json_path)
    trades = load_trades(jsonl_path)
    return market, trades


# collect all json file paths
all_json_paths = [
    json_path
    for folder in folders
    for json_path in sorted(folder.glob("*.json"))
]

all_trades: list[dict] = []
markets: dict[str, dict] = {}

num_workers = min(32, os.cpu_count() * 4 or 16)  # more threads for I/O-bound work

with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = {executor.submit(load_market_and_trades, p): p for p in all_json_paths}
    
    for future in as_completed(futures):
        result = future.result()
        if result is None:
            continue
        
        market, trades = result
        condition_id = market["condition_id"]
        
        # keep only the first time we see a condition_id
        if condition_id not in markets:
            markets[condition_id] = market
        
        all_trades.extend(trades)

print(f"Loaded {len(all_trades):,} trade records across {len(markets):,} unique markets.")

Loaded 112,991 trade records across 721 unique markets.


## 2 · Build the trades DataFrame

In [7]:
df = pd.DataFrame(all_trades)

# parse timestamp → UTC datetime (vectorized)
df["dt"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# canonical column names
df.rename(columns={"proxyWallet": "wallet", "conditionId": "condition_id"}, inplace=True)

# ensure numeric types (use pd.to_numeric for better performance)
df["size"] = pd.to_numeric(df["size"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# sort chronologically (consider if you really need this - it's expensive)
df.sort_values(["condition_id", "wallet", "dt"], inplace=True, ignore_index=True)

print(df.shape)
df.head(3)

(112991, 20)


,wallet,side,asset,condition_id,size,price,timestamp,title,slug,icon,eventSlug,outcome,outcomeIndex,name,pseudonym,bio,profileImage,profileImageOptimized,transactionHash,dt
0,0x27d9c39d57fbd72c0ef92761b589fef4a21a3b92,BUY,1831178517020438850608898839884868461589109173...,0x00127f7be65484384710f01b53e56d59bacf6433c5c8...,7.98,0.94,1772196657,Will Nikola Jokic Rookie Card be above $215 by...,nikola-jokic-rookie-card-above-215-on-february...,https://polymarket-upload.s3.us-east-2.amazona...,nikola-jokic-rookie-card-above-in-february-2026,Yes,0,gklom,Milky-Basil,,,,0x922031dcff0c6b7d52e3a69215fc9e9f63821f5fe980...,2026-02-27 12:50:57+00:00
1,0x27d9c39d57fbd72c0ef92761b589fef4a21a3b92,BUY,1831178517020438850608898839884868461589109173...,0x00127f7be65484384710f01b53e56d59bacf6433c5c8...,5.55,0.94,1772196865,Will Nikola Jokic Rookie Card be above $215 by...,nikola-jokic-rookie-card-above-215-on-february...,https://polymarket-upload.s3.us-east-2.amazona...,nikola-jokic-rookie-card-above-in-february-2026,Yes,0,gklom,Milky-Basil,,,,0x4c58419e4d3cce3da6e66c6a13d7d216b3ae775667ab...,2026-02-27 12:54:25+00:00
2,0x27d9c39d57fbd72c0ef92761b589fef4a21a3b92,BUY,1831178517020438850608898839884868461589109173...,0x00127f7be65484384710f01b53e56d59bacf6433c5c8...,30.00,0.95,1772499459,Will Nikola Jokic Rookie Card be above $215 by...,nikola-jokic-rookie-card-above-215-on-february...,https://polymarket-upload.s3.us-east-2.amazona...,nikola-jokic-rookie-card-above-in-february-2026,Yes,0,gklom,Milky-Basil,,,,0x62783a1bf68dc9b718330fc4bcf1a8a9d2091f71df88...,2026-03-03 00:57:39+00:00


## 3 · Enrich with market metadata and final token value

For each trade we add:

| column | description |
|---|---|
| `question` | Human-readable market question |
| `end_date` | Market resolution date |
| `token_winner` | `True` if the traded token is the winning outcome |
| `final_price` | Final settlement price of the traded token (1.0 = winner, 0.0 = loser, or last known price if unresolved) |
| `trade_value_usdc` | `size × price` — USDC spent on the trade |
| `final_value_usdc` | `size × final_price` — USDC value of shares at resolution |

In [8]:
def build_token_lookup(markets: dict[str, dict]) -> dict[str, dict]:
    """Return {token_id → {condition_id, outcome, winner, final_price}} for all markets."""
    lookup: dict[str, dict] = {}
    for cid, m in markets.items():
        for tok in m.get("tokens", []):
            token_id = str(tok["token_id"])
            winner = bool(tok.get("winner", False))
            # if market is closed/resolved use binary settlement, else use last price
            if m.get("closed", False):
                final_price = 1.0 if winner else 0.0
            else:
                final_price = float(tok.get("price") or 0.0)
            lookup[token_id] = {
                "condition_id": cid,
                "outcome":      tok.get("outcome", ""),
                "token_winner": winner,
                "final_price":  final_price,
            }
    return lookup


token_lookup = build_token_lookup(markets)
print(f"Token lookup entries: {len(token_lookup):,}")

Token lookup entries: 1,442


In [9]:
# join token resolution data
token_df = pd.DataFrame.from_dict(token_lookup, orient="index")
token_df.index.name = "asset"
token_df.reset_index(inplace=True)

df = df.merge(
    token_df[["asset", "token_winner", "final_price"]],
    on="asset",
    how="left",
)

# join market-level metadata
market_meta = pd.DataFrame(
    [
        {
            "condition_id": cid,
            "question":     m["question"],
            "end_date":     pd.to_datetime(m["end_date_iso"], utc=True),
            "market_slug":  m["market_slug"],
        }
        for cid, m in markets.items()
    ]
)

df = df.merge(market_meta, on="condition_id", how="left")

# derived value columns
df["trade_value_usdc"] = df["size"] * df["price"]
df["final_value_usdc"] = df["size"] * df["final_price"]

print(df.shape)
df[["wallet", "condition_id", "dt", "side", "outcome", "size", "price",
    "token_winner", "final_price", "trade_value_usdc", "final_value_usdc"]].head(5)

(112991, 27)


,wallet,condition_id,dt,side,outcome,size,price,token_winner,final_price,trade_value_usdc,final_value_usdc
0,0x27d9c39d57fbd72c0ef92761b589fef4a21a3b92,0x00127f7be65484384710f01b53e56d59bacf6433c5c8...,2026-02-27 12:50:57+00:00,BUY,Yes,7.98,0.94,True,1.0,7.5012,7.98
1,0x27d9c39d57fbd72c0ef92761b589fef4a21a3b92,0x00127f7be65484384710f01b53e56d59bacf6433c5c8...,2026-02-27 12:54:25+00:00,BUY,Yes,5.55,0.94,True,1.0,5.2170,5.55
2,0x27d9c39d57fbd72c0ef92761b589fef4a21a3b92,0x00127f7be65484384710f01b53e56d59bacf6433c5c8...,2026-03-03 00:57:39+00:00,BUY,Yes,30.00,0.95,True,1.0,28.5000,30.00
3,0x27d9c39d57fbd72c0ef92761b589fef4a21a3b92,0x00127f7be65484384710f01b53e56d59bacf6433c5c8...,2026-03-03 12:39:31+00:00,BUY,Yes,6.27,0.97,True,1.0,6.0819,6.27
4,0x299cb40195d270438f8950902f242d796457d85e,0x00127f7be65484384710f01b53e56d59bacf6433c5c8...,2026-03-03 13:01:37+00:00,BUY,Yes,150.00,0.98,True,1.0,147.0000,150.00


## 4 · Group by wallet × market × timestamp

Each row in `grouped` represents all trades made by one wallet in one market at
one exact timestamp (i.e. a single on-chain transaction can place multiple fills).

In [10]:
GROUP_KEYS = ["wallet", "condition_id", "dt"]

grouped = (
    df.groupby(GROUP_KEYS, sort=False)
    .agg(
        question          = ("question",          "first"),
        market_slug       = ("market_slug",        "first"),
        end_date          = ("end_date",           "first"),
        side              = ("side",               "first"),   # BUY / SELL
        outcome           = ("outcome",            "first"),   # Yes / No
        token_winner      = ("token_winner",       "first"),
        final_price       = ("final_price",        "first"),
        # sum across fills in the same tx
        total_size        = ("size",               "sum"),
        avg_price         = ("price",              "mean"),
        trade_value_usdc  = ("trade_value_usdc",   "sum"),
        final_value_usdc  = ("final_value_usdc",   "sum"),
        num_fills         = ("transactionHash",    "count"),
    )
    .reset_index()
    .sort_values(["wallet", "condition_id", "dt"])
    .reset_index(drop=True)
)

print(f"Grouped rows: {len(grouped):,}  (from {len(df):,} raw trades)")
grouped.head(10)

Grouped rows: 107,971  (from 112,991 raw trades)


,wallet,condition_id,dt,question,market_slug,end_date,side,outcome,token_winner,final_price,total_size,avg_price,trade_value_usdc,final_value_usdc,num_fills
0,0x0018938adc5efaa56026842054d02ee6673e78f9,0x0f314aeb3db2e3d769ceaf90c803ca5733a60a9ac97c...,2026-02-28 19:04:21+00:00,Capitals vs. Canadiens,nhl-wsh-mon-2026-02-28,2026-03-01 00:00:00+00:00,BUY,Canadiens,True,1.0,13.114753,0.61,7.999999,13.114753,1
1,0x0018938adc5efaa56026842054d02ee6673e78f9,0x2d107430fd8f23ee3212133f61025eb21be1b8599030...,2026-02-28 19:02:11+00:00,Flames vs. Kings,nhl-cal-lak-2026-02-28,2026-03-01 00:00:00+00:00,BUY,Flames,False,0.0,64.285713,0.42,26.999999,0.000000,1
2,0x0018938adc5efaa56026842054d02ee6673e78f9,0x50327b9a443223686dfae8ddb169d111ee10ddd83d3b...,2026-02-28 19:02:23+00:00,Predators vs. Stars,nhl-nsh-dal-2026-02-28,2026-03-01 00:00:00+00:00,BUY,Predators,False,0.0,62.790696,0.43,26.999999,0.000000,1
3,0x0018938adc5efaa56026842054d02ee6673e78f9,0x50327b9a443223686dfae8ddb169d111ee10ddd83d3b...,2026-02-28 19:02:33+00:00,Predators vs. Stars,nhl-nsh-dal-2026-02-28,2026-03-01 00:00:00+00:00,SELL,Predators,False,0.0,62.790000,0.42,26.371800,0.000000,1
4,0x0018938adc5efaa56026842054d02ee6673e78f9,0x50327b9a443223686dfae8ddb169d111ee10ddd83d3b...,2026-02-28 19:02:47+00:00,Predators vs. Stars,nhl-nsh-dal-2026-02-28,2026-03-01 00:00:00+00:00,BUY,Stars,True,1.0,46.551723,0.58,26.999999,46.551723,1
5,0x0018938adc5efaa56026842054d02ee6673e78f9,0xa7f5f63747c5a43e37d304026791b2935e55c74e85bc...,2026-02-28 19:00:51+00:00,Red Wings vs. Hurricanes,nhl-det-car-2026-02-28,2026-03-01 00:00:00+00:00,BUY,Hurricanes,True,1.0,69.230768,0.65,44.999999,69.230768,1
6,0x0018938adc5efaa56026842054d02ee6673e78f9,0xaaf9e9511e0fd35a5d2a10f3ee7beb6289ef24e66d91...,2026-02-28 19:03:53+00:00,Senators vs. Maple Leafs: O/U 5.5,nhl-ott-tor-2026-02-28-total-5pt5,2026-03-01 00:00:00+00:00,BUY,Over,True,1.0,18.032784,0.61,10.999998,18.032784,1
7,0x0018938adc5efaa56026842054d02ee6673e78f9,0xb57cc844d94ee1758b80331bad3c74b0f458485104b8...,2026-02-28 19:01:17+00:00,Canucks vs. Kraken,nhl-van-sea-2026-02-28,2026-03-01 00:00:00+00:00,BUY,Kraken,True,1.0,45.000000,0.60,27.000000,45.000000,1
8,0x0018938adc5efaa56026842054d02ee6673e78f9,0xccc3172373c5b979785eae70e880a3c2764d63669217...,2026-02-28 19:05:55+00:00,Senators vs. Maple Leafs,nhl-ott-tor-2026-02-28,2026-03-01 00:00:00+00:00,BUY,Senators,True,1.0,10.714284,0.56,5.999999,10.714284,1
9,0x0018938adc5efaa56026842054d02ee6673e78f9,0xfb5a816b39b8bcc90e43d3f11bf303f3ae2454f07c27...,2026-02-28 19:03:23+00:00,Sabres vs. Lightning: O/U 5.5,nhl-buf-tb-2026-02-28-total-5pt5,2026-03-01 00:00:00+00:00,BUY,Over,True,1.0,45.000000,0.60,27.000000,45.000000,1


## 5 · Wallet-level P&L summary (training data only)

Aggregate per wallet using **training trades only** (`dt.date <= END_DATE_TRAIN`).
This ensures the top-5% selection is not contaminated by future (test) data.

In [11]:
# sign convention: BUY spends USDC, SELL receives USDC
grouped["signed_cost"] = grouped.apply(
    lambda r: r["trade_value_usdc"] if r["side"] == "BUY" else -r["trade_value_usdc"],
    axis=1,
)
grouped["signed_final"] = grouped.apply(
    lambda r: r["final_value_usdc"] if r["side"] == "BUY" else -r["final_value_usdc"],
    axis=1,
)

# ── restrict to training period for wallet ranking ────────────────────────────
end_date_train_ts = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train = grouped[grouped["dt"] < end_date_train_ts]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets       = ("condition_id",  "nunique"),
        num_trades        = ("num_fills",      "sum"),
        total_cost_usdc   = ("signed_cost",    "sum"),
        total_final_usdc  = ("signed_final",   "sum"),
    )
    .assign(pnl_usdc=lambda x: x["total_final_usdc"] - x["total_cost_usdc"] )
    .sort_values("pnl_usdc", ascending=False)
    .reset_index()
)

print(f"Unique wallets (training period): {len(wallet_summary):,}")
wallet_summary.head(2)

Unique wallets (training period): 779


,wallet,num_markets,num_trades,total_cost_usdc,total_final_usdc,pnl_usdc
0,0xe79cb0325457e8127071052c0d93a5b10194c715,1,4,1549.999997,3262.928744,1712.928747
1,0x8cb4cd50c5cbc231d5df93c097b605fdf116b997,21,25,4512.096590,5115.340000,603.243410


## 6 · Market-level volume summary

In [12]:
market_summary = (
    grouped.groupby(["condition_id", "question", "end_date", "token_winner"])
    .agg(
        num_wallets      = ("wallet",            "nunique"),
        num_trades       = ("num_fills",          "sum"),
        volume_usdc      = ("trade_value_usdc",   "sum"),
        final_value_usdc = ("final_value_usdc",   "sum"),
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Unique markets: 1,338


,condition_id,question,end_date,token_winner,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x648307c48625289f3655ba5493648e3dffe35850c18f...,Will Austin FC win on 2026-03-01?,2026-03-01 00:00:00+00:00,False,89,288,566315.186575,52.446362
1,0x306d10d4a4d51b41910dbc779ca00908bd917c131541...,US forces enter Iran by March 31?,2026-03-01 00:00:00+00:00,False,1793,3500,483384.997532,494218.058794
2,0x20683c59b213d75abdabf8502f5a053e24266b422db7...,Panthers vs. Islanders,2026-03-01 00:00:00+00:00,False,560,1654,401723.437507,58.586158
3,0x50327b9a443223686dfae8ddb169d111ee10ddd83d3b...,Predators vs. Stars,2026-03-01 00:00:00+00:00,False,447,1278,395698.735217,52.000000
4,0x6d47896d0c6aeeb36eb91b6a281d9b98d13097502c4f...,Sabres vs. Lightning: O/U 6.5,2026-03-01 00:00:00+00:00,True,46,156,393140.733136,773834.837084
5,0x0ff7cc6e6971d0bc567acf0cab6db66e984f176e2a12...,Will Philadelphia Union win on 2026-03-01?,2026-03-01 00:00:00+00:00,True,127,337,387434.134427,645064.593921
6,0x73155755d6eb0ce2a234b76c92352cd9bdd3250a422c...,Blackhawks vs. Utah,2026-03-01 00:00:00+00:00,True,311,756,364737.726152,615829.841329
7,0x228882ef15410cda88ae94b6fee134e87de5360d8439...,Golden Knights vs. Penguins,2026-03-01 00:00:00+00:00,False,487,1113,349383.116408,60.566776
8,0x0b3c5ec8623efb4c12698346af32ced29f2662cfa20d...,Blues vs. Wild,2026-03-01 00:00:00+00:00,False,595,978,293146.685981,0.000000
9,0xbd1d8481b5a809a4806c07e24976effa7b33a6989165...,Golden Knights vs. Penguins: O/U 6.5,2026-03-01 00:00:00+00:00,False,62,98,269745.396327,0.000000


## 7 · Wallet statistics (quantiles)

Distribution of activity and P&L across wallets (training period).

In [13]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "pnl_usdc"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

# number of wallets at or below each quantile threshold
wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])

# append count, mean and std for reference
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "pnl_usdc"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")

wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["num_trades"]   = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"]  = wallet_stats["num_markets"].round(1)
wallet_stats["pnl_usdc"]     = wallet_stats["pnl_usdc"].round(2)

wallet_stats

,wallet_count,num_trades,num_markets,pnl_usdc
quantile,,,,
0.001,1,1.0,1.0,-2025.93
0.01,8,1.0,1.0,-238.12
0.05,39,1.0,1.0,-63.72
0.1,78,1.0,1.0,-59.59
0.25,195,1.0,1.0,-10.00
0.5,390,1.0,1.0,-0.15
0.75,584,3.0,2.0,3.03
0.9,701,8.0,6.0,69.49
0.95,740,19.1,11.1,78.09


## 8 · Full enriched trade table

The main output: one row per wallet × market × timestamp, with all enrichment columns.

In [14]:
DISPLAY_COLS = [
    "wallet", "condition_id", "dt", "end_date",
    "question", "side", "outcome", "total_size", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "num_fills",
]

grouped[DISPLAY_COLS]

,wallet,condition_id,dt,end_date,question,side,outcome,total_size,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,num_fills
0,0x0018938adc5efaa56026842054d02ee6673e78f9,0x0f314aeb3db2e3d769ceaf90c803ca5733a60a9ac97c...,2026-02-28 19:04:21+00:00,2026-03-01 00:00:00+00:00,Capitals vs. Canadiens,BUY,Canadiens,13.114753,0.610000,True,1.00,7.999999,13.114753,1
1,0x0018938adc5efaa56026842054d02ee6673e78f9,0x2d107430fd8f23ee3212133f61025eb21be1b8599030...,2026-02-28 19:02:11+00:00,2026-03-01 00:00:00+00:00,Flames vs. Kings,BUY,Flames,64.285713,0.420000,False,0.00,26.999999,0.000000,1
2,0x0018938adc5efaa56026842054d02ee6673e78f9,0x50327b9a443223686dfae8ddb169d111ee10ddd83d3b...,2026-02-28 19:02:23+00:00,2026-03-01 00:00:00+00:00,Predators vs. Stars,BUY,Predators,62.790696,0.430000,False,0.00,26.999999,0.000000,1
3,0x0018938adc5efaa56026842054d02ee6673e78f9,0x50327b9a443223686dfae8ddb169d111ee10ddd83d3b...,2026-02-28 19:02:33+00:00,2026-03-01 00:00:00+00:00,Predators vs. Stars,SELL,Predators,62.790000,0.420000,False,0.00,26.371800,0.000000,1
4,0x0018938adc5efaa56026842054d02ee6673e78f9,0x50327b9a443223686dfae8ddb169d111ee10ddd83d3b...,2026-02-28 19:02:47+00:00,2026-03-01 00:00:00+00:00,Predators vs. Stars,BUY,Stars,46.551723,0.580000,True,1.00,26.999999,46.551723,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107966,0xffed78a922860b6109cfd2cb1f1448be3d93e122,0x3c7a1d094937de0f72e1fce48badb82cac6fb5451fc0...,2026-02-10 16:08:05+00:00,2026-03-01 00:00:00+00:00,Will Fnatic win the LEC 2026 Versus?,BUY,Yes,7.180000,0.029000,False,0.00,0.208220,0.000000,1
107967,0xffed78a922860b6109cfd2cb1f1448be3d93e122,0x66d3b4da447553561c5f62fd031a40b7640229b427d0...,2026-02-10 16:07:59+00:00,2026-03-01 00:00:00+00:00,Will Team Heretics win the LEC 2026 Versus?,BUY,Yes,25.000000,0.040000,False,0.00,1.000000,0.000000,1
107968,0xffefd9de2f94441c5ddec96358a62695a37db71e,0x228882ef15410cda88ae94b6fee134e87de5360d8439...,2026-03-01 19:21:55+00:00,2026-03-01 00:00:00+00:00,Golden Knights vs. Penguins,BUY,Penguins,43.010742,0.930000,True,1.00,39.999990,43.010742,1
107969,0xfff7b763c931a7e677c2589ce651f91a1c759947,0x1af30b5eedc5249fe8cebbccba0be0a20be4fa0507fd...,2026-02-05 07:47:21+00:00,2026-03-01 00:00:00+00:00,Will NVIDIA reach $272 in February?,BUY,No,1.003000,0.997000,True,1.00,0.999991,1.003000,1


## 9 · Export top-5% wallets to parquet

Identifies top-5% wallets by P&L on the **training period**, then exports their
trades from the **full dataset** (training + test) to `data/trades_processed.parquet`.

Each row is a single trade fill enriched with `trade_value_usdc`, `final_value_usdc`,
and a boolean `is_train` flag that marks whether the trade falls in the training window.

In [15]:
# ── identify top-5% wallets by training-period P&L ───────────────────────────
pnl_threshold = wallet_summary["pnl_usdc"].quantile(0.95)
top_wallets   = set(wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "wallet"])
print(f"P&L threshold (95th pct, training): {pnl_threshold:,.2f} USDC")
print(f"Top-5% wallet count:                {len(top_wallets):,}")

# print deciles of training P&L for top wallets
for i in range(0, 11):
    q = i / 10
    pnl_q = wallet_summary.loc[wallet_summary["pnl_usdc"] >= pnl_threshold, "pnl_usdc"].quantile(q)
    print(f"  P&L at {q:.0%} pct: {pnl_q:,.2f} USDC")

P&L threshold (95th pct, training): 78.09 USDC
Top-5% wallet count:                39
  P&L at 0% pct: 78.10 USDC
  P&L at 10% pct: 78.38 USDC
  P&L at 20% pct: 78.79 USDC
  P&L at 30% pct: 79.25 USDC
  P&L at 40% pct: 81.77 USDC
  P&L at 50% pct: 108.24 USDC
  P&L at 60% pct: 120.38 USDC
  P&L at 70% pct: 131.73 USDC
  P&L at 80% pct: 171.45 USDC
  P&L at 90% pct: 236.80 USDC
  P&L at 100% pct: 1,712.93 USDC


In [16]:
EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "asset", "outcome", "outcomeIndex",
    "size", "price",
    "trade_value_usdc", "final_value_usdc",
    "token_winner", "final_price",
    "name", "pseudonym", "transactionHash", "title",
    "is_train",
]

# ── filter full dataset (all dates) to top wallets ────────────────────────────
top_trades = df[df["wallet"].isin(top_wallets)].copy()

# ── flag training vs test rows ────────────────────────────────────────────────
top_trades["is_train"] = top_trades["dt"].dt.date <= END_DATE_TRAIN

print(f"Total trade rows:  {len(top_trades):,}")
print(f"  training rows:   {top_trades['is_train'].sum():,}")
print(f"  test rows:       {(~top_trades['is_train']).sum():,}")

# ── write to parquet ──────────────────────────────────────────────────────────
TRADES_PARQUET.parent.mkdir(parents=True, exist_ok=True)
top_trades[EXPORT_COLS].to_parquet(TRADES_PARQUET, index=False)
print(f"\nSaved {len(top_trades):,} rows → {TRADES_PARQUET}")

Total trade rows:  3,789
  training rows:   2,045
  test rows:       1,744

Saved 3,789 rows → ../data/trades_processed.parquet


In [21]:
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

In [22]:
df.sample(5)

,wallet,side,asset,condition_id,size,price,timestamp,title,slug,icon,...,profileImageOptimized,transactionHash,dt,token_winner,final_price,question,end_date,market_slug,trade_value_usdc,final_value_usdc
63819,0x65f700612f89853c6c36684d3d3da5ee3de3ff15,SELL,5945331088240115062921102208238534558998618518...,0x73155755d6eb0ce2a234b76c92352cd9bdd3250a422c...,1.850000,0.930000,2026-03-01 23:01:29+00:00,Blackhawks vs. Utah,nhl-chi-utah-2026-03-01,https://polymarket-upload.s3.us-east-2.amazona...,...,,0x7d1e577176102421b910fe6b1b1cd0101687b3bf2b5b...,2026-03-01 23:01:29+00:00,True,1.00,Blackhawks vs. Utah,2026-03-01 00:00:00+00:00,nhl-chi-utah-2026-03-01,1.720500,1.850000
33874,0xf8647baaad4f49cbf6fad8cf0d26f7b8c5f204c2,BUY,8169748624039290189916764999700873638013791190...,0x306d10d4a4d51b41910dbc779ca00908bd917c131541...,1.492536,0.670000,2026-03-06 13:47:33+00:00,US forces enter Iran by March 31?,us-forces-enter-iran-by-march-31-222-191-243-5...,https://polymarket-upload.s3.us-east-2.amazona...,...,,0x710fff9f761d48f5bef828dbefb36156040697cd9a5f...,2026-03-06 13:47:33+00:00,False,0.59,US forces enter Iran by March 31?,2026-03-01 00:00:00+00:00,us-forces-enter-iran-by-march-31-222-191-243-5...,0.999999,0.880596
84450,0x204f72f35326db932158cba6adff0b9a1da95e14,BUY,2856420079681608014796106877264344358369610054...,0xae0e985eaeae527516a9326b0762b319c2c50725aecb...,144.000000,0.240000,2026-02-28 23:39:27+00:00,Real Salt Lake vs. Seattle Sounders FC: O/U 1.5,mls-rsl-sea-2026-02-28-total-1pt5,https://polymarket-upload.s3.us-east-2.amazona...,...,,0xb43d616ad44f6387f79892fb51cfb90cc45f49a789a0...,2026-02-28 23:39:27+00:00,False,0.00,Real Salt Lake vs. Seattle Sounders FC: O/U 1.5,2026-03-01 00:00:00+00:00,mls-rsl-sea-2026-02-28-total-1pt5,34.560000,0.000000
28880,0x866637ca2de80c0dda240e9e334c454fda70a368,BUY,4465136166232039922652880617937191319155191950...,0x2d107430fd8f23ee3212133f61025eb21be1b8599030...,9.980000,0.420000,2026-02-28 11:05:19+00:00,Flames vs. Kings,nhl-cal-lak-2026-02-28,https://polymarket-upload.s3.us-east-2.amazona...,...,,0x66c3136e43e00a8134f031e295434e90c6291783ad84...,2026-02-28 11:05:19+00:00,False,0.00,Flames vs. Kings,2026-03-01 00:00:00+00:00,nhl-cal-lak-2026-02-28,4.191600,0.000000
73271,0x42b279b94eb8404095b0b1f9ffcd9038461c00c3,BUY,4891654006883092051823378342158985447293459010...,0x93338912a28038645e164a173eab0ee29aaefa75b1a4...,10.340000,0.996898,2026-01-18 16:21:26+00:00,Will Natus Vincere win the LEC 2026 Versus?,will-natus-vincere-win-the-lec-2026-versus,https://polymarket-upload.s3.us-east-2.amazona...,...,,0x2f60cc716c6872fef6afe736278ab5255354807fac21...,2026-01-18 16:21:26+00:00,True,1.00,Will Natus Vincere win the LEC 2026 Versus?,2026-03-01 00:00:00+00:00,will-natus-vincere-win-the-lec-2026-versus,10.307930,10.340000


In [26]:
agg = (
    df.groupby("wallet", as_index=False)
      .agg(count=("wallet", "count"))
      .sort_values("count", ascending=False)
      .reset_index(drop=True)
)

In [30]:
pd.concat([agg.head(5), agg.tail(15)])

,wallet,count
0,0x204f72f35326db932158cba6adff0b9a1da95e14,3829
1,0xada56a3f1a9d7d3fe7ec8cbcaa0865109c57db70,2215
2,0x38d812aff0b79f3bf5da2a477f780bcc163eea7c,2097
3,0xe3726a1b9c6ba2f06585d1c9e01d00afaedaeb38,1587
4,0xeface9902242b0399856f981600b907dcc9bc9a1,1399
13989,0x85facdd9123f78f6391c0ab88a35d89825dc2c45,1
13990,0x85f38358b8c55bd9e697f3f4e355ed6524525a50,1
13991,0x85e4c8ee85015e6762df71cf81f54e5882cd681d,1
13992,0x85e0c721e18b8b84d20a2ea3d2a1d031eff69f06,1
13993,0x85d82cc5fcb23983adb21594827d4f347a323cbe,1
